# ⚡ Long Range Arena (LRA) Benchmark - Piecewise Linear Attention

Benchmark **piecewise linear attention** on the Long Range Arena benchmark suite.

## What This Does

- Compares **Standard**, **Linear**, and **Piecewise** attention on LRA tasks
- Currently supports **ListOps** task (hierarchical list operations)
- Measures accuracy, training time, and speedup
- Generates comparison tables and visualizations

## Expected Runtime

- **Quick test (2 epochs)**: ~10-15 minutes on T4 GPU
- **Full benchmark (20 epochs)**: ~1.5-2 hours on T4 GPU
- **Full benchmark (20 epochs)**: ~45-60 minutes on T100 GPU

---

## 📦 Setup

In [ ]:
# Configuration
REPO_URL = "https://github.com/grapentt/piecewise-linear-attention.git"
BRANCH = "main"  # Change to your branch if needed

# Clone repository
import os
if not os.path.exists("piecewise-linear-attention"):
    print(f"Cloning repository (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already exists")

%cd piecewise-linear-attention
!git pull origin {BRANCH}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install torch tqdm numpy -q
print("✓ Installation complete!")

In [ ]:
# Check device
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠ No GPU - using CPU (slower but still works)")

print(f"\nDevice: {device}")

---

## 🎯 Benchmark Configuration

In [ ]:
# Benchmark settings
EPOCHS = 20  # Change to 2 for quick test
BATCH_SIZE = 32  # Reduce to 16 if out of memory
ATTENTION_TYPES = ["standard", "linear", "piecewise"]  # Or just ["piecewise"] for quick test
NUM_WORKERS = 2  # Data loading workers

print("Benchmark Configuration:")
print(f"  Task: ListOps")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Attention types: {', '.join(ATTENTION_TYPES)}")
print(f"  Device: {device}")

---

## 📊 Generate Dataset

ListOps dataset will be generated locally (GCS bucket has restricted access).

In [ ]:
# Dataset is generated automatically on first run
# This cell just confirms the data directory exists
!mkdir -p data/lra/listops
print("✓ Data directory ready")
print("  Dataset will be generated automatically during training if not present")

---

## 🚀 Run Benchmark

In [ ]:
# Run benchmark
print("\n" + "="*80)
print("STARTING LRA LISTOPS BENCHMARK")
print("="*80 + "\n")

!python experiments/lra/tasks/listops/train.py \
  --attention-types {" ".join(ATTENTION_TYPES)} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --device {device} \
  --num-workers {NUM_WORKERS} \
  --output-dir results/lra/listops

print("\n" + "="*80)
print("✓ BENCHMARK COMPLETE!")
print("="*80)

---

## 📈 Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("results/lra/listops/results.json") as f:
    results = json.load(f)

print(f"Loaded results for {len(results)} attention types")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {'standard': '#e74c3c', 'linear': '#3498db', 'piecewise': '#2ecc71'}

# Plot 1: Training accuracy
ax = axes[0]
for attn_type, data in results.items():
    train_history = data['train_history']
    epochs = list(range(1, len(train_history) + 1))
    accuracies = [h['accuracy'] * 100 for h in train_history]
    ax.plot(epochs, accuracies, 'o-', linewidth=2, markersize=6, 
            label=attn_type.capitalize(), color=colors.get(attn_type, 'gray'))

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Training Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Training Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Plot 2: Validation accuracy
ax = axes[1]
for attn_type, data in results.items():
    val_history = data['val_history']
    epochs = list(range(1, len(val_history) + 1))
    accuracies = [h['accuracy'] * 100 for h in val_history]
    ax.plot(epochs, accuracies, 's-', linewidth=2, markersize=6, 
            label=attn_type.capitalize(), color=colors.get(attn_type, 'gray'))

# Add LRA baseline
ax.axhline(y=36.37, color='red', linestyle='--', alpha=0.5, linewidth=2, label='LRA Baseline')

ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax.set_ylabel('Validation Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Validation Progress', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

fig.suptitle('LRA ListOps - Training Curves', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

attention_types = list(results.keys())
val_accs = [results[at]['best_val_accuracy'] * 100 for at in attention_types]
test_accs = [results[at]['test_metrics']['accuracy'] * 100 for at in attention_types]
times = [results[at]['total_time'] / 60 for at in attention_types]  # Convert to minutes

x = np.arange(len(attention_types))
width = 0.35

# Plot 1: Accuracies
ax = axes[0]
ax.bar(x - width/2, val_accs, width, label='Val Accuracy', color='#3498db', alpha=0.8)
ax.bar(x + width/2, test_accs, width, label='Test Accuracy', color='#2ecc71', alpha=0.8)
ax.axhline(y=36.37, color='red', linestyle='--', alpha=0.5, linewidth=2)
ax.text(len(attention_types) - 0.5, 36.37 + 0.5, 'LRA Baseline', 
        fontsize=10, color='red', ha='right')

ax.set_ylabel('Accuracy (%)', fontsize=12, fontweight='bold')
ax.set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([at.capitalize() for at in attention_types], fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Training time
ax = axes[1]
bar_colors = [colors.get(at, 'gray') for at in attention_types]
ax.bar(x, times, color=bar_colors, alpha=0.8)
ax.set_ylabel('Training Time (minutes)', fontsize=12, fontweight='bold')
ax.set_title('Training Time Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([at.capitalize() for at in attention_types], fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

# Add time labels on bars
for i, time in enumerate(times):
    ax.text(i, time + max(times) * 0.02, f"{time:.1f}m", 
            ha='center', fontsize=10, fontweight='bold')

fig.suptitle('LRA ListOps - Performance Summary', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

In [ ]:
# Speedup analysis
if 'standard' in results:
    baseline_time = results['standard']['total_time']
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    speedups = []
    speedup_labels = []
    
    for attn_type in attention_types:
        if attn_type != 'standard':
            speedup = baseline_time / results[attn_type]['total_time']
            speedups.append(speedup)
            speedup_labels.append(attn_type.capitalize())
    
    x = np.arange(len(speedup_labels))
    bar_colors = [colors.get(label.lower(), 'gray') for label in speedup_labels]
    bars = ax.bar(x, speedups, color=bar_colors, alpha=0.8)
    
    ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, linewidth=2, label='Baseline')
    ax.set_ylabel('Speedup vs Standard', fontsize=12, fontweight='bold')
    ax.set_title('Training Speedup (Higher is Better)', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(speedup_labels, fontsize=11)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add speedup labels on bars
    for i, speedup in enumerate(speedups):
        ax.text(i, speedup + max(speedups) * 0.02, f"{speedup:.2f}×", 
                ha='center', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("\n" + "="*80)
    print("SPEEDUP ANALYSIS")
    print("="*80)
    for label, speedup in zip(speedup_labels, speedups):
        print(f"  {label}: {speedup:.2f}× faster than Standard")
    print("="*80)
else:
    print("Note: Speedup analysis requires 'standard' attention to be benchmarked")

---

## 📋 Detailed Results Table

In [ ]:
import pandas as pd

# Create summary table
df_data = []
lra_baseline = 36.37

baseline_time = results.get('standard', {}).get('total_time', None)

for attn_type in attention_types:
    data = results[attn_type]
    val_acc = data['best_val_accuracy'] * 100
    test_acc = data['test_metrics']['accuracy'] * 100
    time_min = data['total_time'] / 60
    
    # Calculate speedup
    if baseline_time and attn_type != 'standard':
        speedup = baseline_time / data['total_time']
        speedup_str = f"{speedup:.2f}×"
    else:
        speedup_str = "baseline" if attn_type == 'standard' else "-"
    
    # Calculate diff from LRA baseline
    diff = test_acc - lra_baseline
    sign = "+" if diff >= 0 else ""
    
    df_data.append({
        "Method": attn_type.capitalize(),
        "Val Acc (%)": f"{val_acc:.2f}",
        "Test Acc (%)": f"{test_acc:.2f}",
        "vs LRA Baseline": f"{sign}{diff:.2f}",
        "Time (min)": f"{time_min:.1f}",
        "Speedup": speedup_str,
    })

df = pd.DataFrame(df_data)
print("\nLRA ListOps Benchmark Results:")
print("="*80)
print(df.to_string(index=False))
print("="*80)
print(f"\nLRA Transformer Baseline: {lra_baseline:.2f}%")
print(f"Dataset: 96K train, 2K val, 2K test samples")
print(f"Sequence length: ~500-2000 tokens (character-level)")
print(f"Model: 4 layers, 8 heads, 512 hidden dim, 1024 MLP dim")
print("="*80)

---

## 💾 Download Results

In [ ]:
from google.colab import files

print("Downloading results...")
files.download("results/lra/listops/results.json")
print("✓ Download complete!")

---

## 🎛️ Advanced: Custom Configuration

Run a custom benchmark with specific settings.

In [ ]:
# Example: Quick test with just piecewise attention (2 epochs)
# Uncomment and run:

# !python experiments/lra/tasks/listops/train.py \
#   --attention-types piecewise \
#   --epochs 2 \
#   --batch-size 16 \
#   --device cuda \
#   --num-workers 2 \
#   --output-dir results/lra/listops_quick

---

## 📚 Learn More

- [Main README](https://github.com/grapentt/piecewise-linear-attention/blob/main/README.md)
- [Theory](https://github.com/grapentt/piecewise-linear-attention/blob/main/THEORY.md)
- [LRA Implementation Plan](https://github.com/grapentt/piecewise-linear-attention/blob/main/experiments/lra/README.md)
- [Long Range Arena Paper](https://openreview.net/forum?id=qVyeW-grC2k)

**Happy Benchmarking! 🚀**